# ส่วนที่ 4 · Fine-tune mT5 สรุปข่าวภาษาไทย (Abstractive)

**ผู้รับผิดชอบ:** คนที่ 4 (คู่ C) · **ส่งมอบงาน:** โมเดลที่ดีที่สุด (`mt5-thaisum`) + ผลคะแนน ROUGE

---

### 📌 วัตถุประสงค์และการทำงาน
โน้ตบุ๊กนี้ใช้สำหรับการฝึกฝนโมเดลสร้างข้อความสรุปข่าวภาษาไทยแบบ **Abstractive Summarization** (โมเดลอ่านเนื้อข่าวทั้งหมดแล้วเรียบเรียงสรุปใหม่ด้วยภาษาของตัวเอง) โดยใช้สถาปัตยกรรม **mT5 (Multilingual T5)** ที่ผ่านการ Pre-train ข่าวภาษาไทยมาแล้วจากชุดข้อมูล **XL-Sum** (`csebuetnlp/mT5_multilingual_XLSum`) แล้วนำมาทำ Transfer Learning / Fine-tuning ต่อด้วยคลังข่าวภาษาไทยคุณภาพสูง **ThaiSum**

**แนวทางการรัน:**
* **แนะนำที่สุด (สำหรับเทรนโมเดลใหม่):** อัปโหลดไฟล์นี้ไปรันบน **Google Colab** โดยเลือก:  
  เมนู `Runtime` → `Change runtime type` → เลือก **T4 GPU** แล้วคลิก `Runtime` → **Run all**
* **สำหรับการทดสอบ/ตรวจสอบบนเครื่องทั่วไป (CPU):**  
  ระบบมี **Smart Fallback** ในตัว หากตรวจพบว่ามีโมเดลและไฟล์ทำนายอยู่แล้ว จะโหลดมาแสดงผลและทดสอบสรุปข่าวได้ทันทีโดยไม่ติดขัดและไม่เกิดข้อผิดพลาด (`NameError`)

---


In [1]:
!pip -q install transformers datasets sentencepiece accelerate rouge_score pythainlp

## 1. การตั้งค่าไฮเปอร์พารามิเตอร์ (Hyperparameters)
อธิบายตัวแปรสำคัญ:
- `MODEL_NAME`: โมเดลตั้งต้นที่ใช้เทรน (เริ่มต้นด้วย `csebuetnlp/mT5_multilingual_XLSum` ขนาด ~580M parameters)
- `TRAIN_SIZE`, `VAL_SIZE`, `TEST_SIZE`: จำนวนข่าวที่ใช้สำหรับเทรน ตรวจสอบ และทดสอบชุดมาตรฐาน
- `MAX_INPUT` (512 token): ความยาวเนื้อข่าวสูงสุดที่ตัดส่งเข้า Encoder ของโมเดล
- `MAX_TARGET` (128 token): ความยาวของบทสรุปข่าวสูงสุดที่ Decoder สร้างออกมา (~2–3 ประโยคภาษาไทย)
- `BATCH_SIZE` (4) และ `GRAD_ACC` (4): ทำให้มี Effective Batch Size = 4 x 4 = 16 ตัวอย่างต่อรอบ เพื่อความเสถียรและประหยัด VRAM บน GPU
- `NUM_BEAMS` (4): จำนวน Beam สำหรับ Beam Search Decoding ตอนสร้างบทสรุป เพื่อคัดเลือกข้อความที่มีความน่าจะเป็นสูงสุด

In [2]:
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"   # หรือ "google/mt5-small" (เริ่มจากศูนย์ ผลแย่กว่า แต่เร็ว)
# mBART (คนที่ 5): "facebook/mbart-large-50" + ตั้ง MBART_LANG = "th_TH"
MBART_LANG = None

# ข้อมูล: None = โหลด ThaiSum จาก Hugging Face · ใส่ path = ใช้ไฟล์ของคนที่ 1
TRAIN_CSV, VAL_CSV, TEST_CSV = None, None, None
TRAIN_SIZE = 5000    # จำนวนข่าวที่ใช้เทรน (Colab T4 ~5,000 ข่าว/epoch ประมาณ 30–40 นาที)
VAL_SIZE   = 300
TEST_SIZE  = 500     # ทุกโมเดลต้องใช้ test ชุดเดียวกัน

MAX_INPUT  = 512     # ความยาวข่าว (token) ที่ป้อนเข้า
MAX_TARGET = 128     # ความยาวบทสรุปสูงสุด
EPOCHS     = 2
LR         = 1e-4    # mt5-small เริ่มจากศูนย์ใช้ 5e-4 · mBART ใช้ 3e-5
BATCH_SIZE = 4
GRAD_ACC   = 4       # batch จริง = 4 x 4 = 16
NUM_BEAMS  = 4
OUTPUT_DIR = "mt5-thaisum"
SEED = 42

## 2. โหลดและจัดการข้อมูล (Data Ingestion & Splitting)
* ใช้ชุดข้อมูล **[ThaiSum](https://huggingface.co/datasets/pythainlp/thaisum)** ซึ่งรวบรวมข่าวจากสำนักข่าวชั้นนำของไทย (ไทยรัฐ, ไทยพีบีเอส, ประชาไท, The Standard)
* โหลดในรูปแบบ **Streaming** ช่วยประหยัดเวลาและหน่วยความจำ ไม่ต้องดาวน์โหลดไฟล์ขนาดหลายกิกะไบต์ทั้งหมด
* คอลัมน์ที่ใช้:
  * `body`: เนื้อข่าวต้นฉบับภาษาไทย
  * `summary`: บทสรุปข่าวมาตรฐานที่เขียนโดยนักข่าวมืออาชีพ (Ground Truth Reference)

In [3]:
import pandas as pd
from datasets import load_dataset, Dataset

def from_thaisum(split, n, shuffle=False):
    ds = load_dataset("pythainlp/thaisum", split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)   # สุ่มให้ได้ข่าวหลายสำนัก
    return pd.DataFrame(list(ds.take(n)))[["body", "summary"]]

def load(csv, split, n, shuffle=False):
    df = pd.read_csv(csv) if csv else from_thaisum(split, n, shuffle)
    df = df.dropna(subset=["body", "summary"])
    return df[(df.body.str.len() > 0) & (df.summary.str.len() > 0)].reset_index(drop=True)

train_df = load(TRAIN_CSV, "train", TRAIN_SIZE, shuffle=True)
val_df   = load(VAL_CSV, "validation", VAL_SIZE)
test_df  = load(TEST_CSV, "test", TEST_SIZE)
print("train", len(train_df), "| val", len(val_df), "| test", len(test_df))
print("\nตัวอย่างข่าว:", train_df.body[0][:200], "...\nบทสรุป:", train_df.summary[0])

d:\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train 5000 | val 300 | test 500

ตัวอย่างข่าว: หลังจากที่ แทค ภรัณยู ควงแฟนสาวนอกวงการ เบลล์ อดิราภ์ เองตระกูล วิวาห์ฟ้าแลบเมื่อวันอังคารที่ผ่านมา จนกลายเป็นข่าวฮือฮาที่หลายคนให้ความสนใจ และนอกจากจะสละโสดแล้ว หนุ่มแทคยังมีข่าวดีอีกเรื่องคือ ไฮโซเบ ...
บทสรุป: มดดำ เผย แทค ภรัณยู คบไฮโซเบลล์ มานานแล้ว ก่อนที่จะคบกับ ดีเจต้นหอม พร้อมบอกฝ่ายหญิงอดทนหนักมาก แม้ว่า แทคจะไปเรื่อยๆ กับคนอื่น


## 3. การตัดคำและแปลงข้อความเป็นตัวเลข (Tokenization)
* ใช้ **SentencePiece Tokenizer** ของ mT5 ที่รองรับอักขระภาษาไทยและหลายภาษาทั่วโลก
* กำหนดขนาดความยาวสูงสุดของ Input (`MAX_INPUT=512`) และ Output (`MAX_TARGET=128`) พร้อมตัดทอน (`truncation=True`) สำหรับข่าวที่ยาวเกินไป
* สร้าง `Dataset` สำหรับ Train และ Validation เพื่อส่งให้โมเดลประมวลผลได้อย่างรวดเร็ว

In [4]:
from transformers import AutoTokenizer
tok_kwargs = {"src_lang": MBART_LANG, "tgt_lang": MBART_LANG} if MBART_LANG else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)

def prep(batch):
    x = tokenizer(batch["body"], max_length=MAX_INPUT, truncation=True)
    x["labels"] = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return x

train_ds = Dataset.from_pandas(train_df).map(prep, batched=True, remove_columns=["body", "summary"])
val_ds   = Dataset.from_pandas(val_df).map(prep, batched=True, remove_columns=["body", "summary"])

lens = [len(tokenizer(b).input_ids) for b in train_df.body[:200]]
print(f"ความยาวข่าวเฉลี่ย {sum(lens)/len(lens):.0f} token · ถูกตัดเกิน {MAX_INPUT}: {sum(l > MAX_INPUT for l in lens)/len(lens):.0%}")

Map: 100%|██████████| 300/300 [00:00<00:00, 322.17 examples/s]


ความยาวข่าวเฉลี่ย 704 token · ถูกตัดเกิน 512: 45%


## 4. ตัววัดประสิทธิภาพ ROUGE สำหรับภาษาไทย (Evaluation Metric)
* การวัดคุณภาพของบทสรุป (Text Summarization) จะใช้คะแนน **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**:
  * **ROUGE-1 (F1):** วัดความตรงกันของคำเดี่ยว (Unigram)
  * **ROUGE-2 (F1):** วัดความตรงกันของคำคู่ติดกัน (Bigram) สะท้อนความลื่นไหลของวลี
  * **ROUGE-L (F1):** วัดลำดับคำที่ยาวที่สุดที่ตรงกัน (Longest Common Subsequence) สะท้อนโครงสร้างประโยค
* ⚠️ **ข้อควรระวังสำคัญสำหรับภาษาไทย:** ภาษาไทยไม่มีการเว้นวรรคระหว่างคำ จึงจำเป็นต้องตัดคำด้วย **PyThaiNLP (เอนจิน `newmm`)** ก่อนคำนวณคะแนน ROUGE ทุกครั้ง (กติการ่วมของทั้ง 8 คนในโปรเจกต์)

In [5]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]}

print(thai_rouge(["นายกรัฐมนตรีประชุมครม.วันนี้"], ["วันนี้นายกรัฐมนตรีเป็นประธานประชุมครม."]))

{'rouge1': 80.0, 'rouge2': 25.0, 'rougeL': 60.0}


## 5. การทดสอบก่อนเทรน (Zero-shot Baseline)
* รันการทดสอบโมเดลเริ่มต้นบน 100 ข่าวแรกของ Test Set ก่อนเริ่มทำการ Fine-tune
* เพื่อบันทึกเป็นคะแนนพื้นฐาน (Baseline) สำหรับใช้วัดว่าการเทรนด้วยข้อมูลข่าวไทย ช่วยเพิ่มความแม่นยำในการสรุปข่าวได้มากน้อยเพียงใด

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")

def summarize(texts, model, bs=8):
    out = []
    model.eval()
    gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id[MBART_LANG]} if MBART_LANG else {}
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS,
                                 no_repeat_ngram_size=3, length_penalty=1.0, **gen_kwargs)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
N0 = min(100, len(test_df))
zero = summarize(test_df.body[:N0].tolist(), model)
print("zero-shot ROUGE (100 ข่าวแรกของ test):", thai_rouge(zero, test_df.summary[:N0].tolist()))
print("\nตัวอย่าง:", zero[0])

GPU: ไม่มี (ช้ามาก ให้เปิด T4 GPU)


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 9016.25it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=84) seem to have been set. `max_new_tokens` will t

## 6. การฝึกฝนโมเดล (Fine-tuning Process)
* **กลยุทธ์การเทรนให้มีประสิทธิภาพและประหยัดหน่วยความจำ:**
  * ใช้ **Adafactor Optimizer** แทน AdamW เพื่อประหยัด VRAM ได้มหาศาล
  * เปิดใช้ **Gradient Checkpointing** ช่วยลดการใช้หน่วยความจำขณะคำนวณ Gradient
  * ใช้การประมวลผลแบบ fp32 (สำหรับ mT5 ซึ่งมักเจอปัญหาค่า Loss กลายเป็น NaN หากใช้ fp16)
  * ตั้งค่า `load_best_model_at_end=True` โดยวัดจาก `eval_loss` ที่ต่ำที่สุด
* **ระบบ Smart Fallback:**  
  หากรันบนเครื่องคอมพิวเตอร์ที่ไม่มี GPU (CPU) และมีโมเดลที่เคยบันทึกไว้ในโฟลเดอร์ `mt5-thaisum/` ระบบจะข้ามขั้นตอนการเทรนให้อัตโนมัติ เพื่อไม่ให้เสียเวลารอนานหลายสิบชั่วโมง

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, set_seed
import os
set_seed(SEED)

# ตรวจสอบว่ามีโมเดลที่เคยเทรน/บันทึกไว้แล้วหรือไม่
already_trained = os.path.exists(f"{OUTPUT_DIR}/config.json") or os.path.exists(f"../{OUTPUT_DIR}/config.json")

if already_trained and device == "cpu":
    print(f"⚡ ตรวจพบโมเดลที่เทรนเสร็จแล้วใน '{OUTPUT_DIR}' (รันบน CPU ข้ามขั้นตอนการเทรนเพื่อประหยัดเวลา)")
    trainer = None
else:
    args = Seq2SeqTrainingArguments(
        output_dir="checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01,
        optim="adafactor", gradient_checkpointing=True,   # ประหยัดหน่วยความจำ GPU
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        fp16=False, logging_steps=50, report_to="none", seed=SEED)

    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
    trainer.train()

⚡ ตรวจพบโมเดลที่เทรนเสร็จแล้วใน 'mt5-thaisum' (รันบน CPU ข้ามขั้นตอนการเทรนเพื่อประหยัดเวลา)


## 7. การวัดผลบนชุดทดสอบ (Test Set Evaluation)
* ส่งมอบผลคะแนนในตารางนี้ให้กับ **คนที่ 7** (สรุปผลและรายงานเปรียบเทียบ)
* สรุปข่าวทั้ง 500 ข่าวใน Test Set โดยใช้โมเดลที่ผ่านการ Fine-tune ด้วยพารามิเตอร์:
  * `num_beams=4` (Beam Search เพื่อเลือกคำตอบที่ดีที่สุด)
  * `no_repeat_ngram_size=3` (ป้องกันไม่ให้โมเดลสร้างคำหรือวลีซ้ำๆ ซากๆ)
* เปรียบเทียบคะแนนกับ Zero-shot ก่อนเทรน

In [ ]:
import os, json
import pandas as pd
from transformers import AutoModelForSeq2SeqLM

# 1. กำหนดโมเดลสำหรับทดสอบ (ใช้ trainer.model หากเพิ่งเทรนเสร็จ หรือโหลดจาก OUTPUT_DIR)
if 'trainer' in globals() and trainer is not None and hasattr(trainer, 'model'):
    eval_model = trainer.model
else:
    model_dir = OUTPUT_DIR if os.path.exists(f"{OUTPUT_DIR}/config.json") else (f"../{OUTPUT_DIR}" if os.path.exists(f"../{OUTPUT_DIR}/config.json") else MODEL_NAME)
    print(f"📦 โหลดโมเดลสำหรับทดสอบจาก: {model_dir}")
    eval_model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)

# 2. ป้องกัน NameError หากรันข้ามเซลล์
if 'summarize' not in globals():
    def summarize(texts, model, bs=8):
        out = []
        model.eval()
        gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id[MBART_LANG]} if MBART_LANG else {}
        for i in range(0, len(texts), bs):
            enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
            with torch.no_grad():
                ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS,
                                     no_repeat_ngram_size=3, length_penalty=1.0, **gen_kwargs)
            out += tokenizer.batch_decode(ids, skip_special_tokens=True)
        return out

# 3. โหลดผลลัพธ์การทำนาย (หากมี predictions_mt5.csv อยู่แล้ว โหลดทันทีไม่ต้องรอคำนวณซ้ำ)
pred_file = "predictions_mt5.csv" if os.path.exists("predictions_mt5.csv") else ("../predictions_mt5.csv" if os.path.exists("../predictions_mt5.csv") else None)
if pred_file:
    print(f"✓ โหลดผลสรุปที่มีอยู่แล้วจาก {pred_file}")
    _df = pd.read_csv(pred_file)
    preds = _df["pred"].tolist()
    refs = _df["summary"].tolist()
    if 'test_df' not in globals():
        test_df = _df
else:
    preds = summarize(test_df.body.tolist(), eval_model)
    refs = test_df.summary.tolist()

# 4. โหลดหรือคำนวณคะแนน ROUGE
res_file = f"{OUTPUT_DIR}/result.json" if os.path.exists(f"{OUTPUT_DIR}/result.json") else (f"../{OUTPUT_DIR}/result.json" if os.path.exists(f"../{OUTPUT_DIR}/result.json") else None)
if res_file:
    with open(res_file, encoding="utf-8") as f:
        result = json.load(f)
else:
    scores = thai_rouge(preds, refs)
    zero_scores = thai_rouge(zero, refs[:N0]) if ('zero' in globals() and 'N0' in globals()) else {}
    train_count = len(train_df) if 'train_df' in globals() else TRAIN_SIZE
    test_count = len(test_df) if 'test_df' in globals() else len(refs)
    result = {"model": MODEL_NAME, **scores, "zero_shot_rouge_first100": zero_scores,
              "train_size": train_count, "test_size": test_count, "epochs": EPOCHS, "lr": LR,
              "max_input": MAX_INPUT, "num_beams": NUM_BEAMS}
result

📦 โหลดโมเดลสำหรับทดสอบจาก: mt5-thaisum
✓ โหลดผลสรุปที่มีอยู่แล้วจาก predictions_mt5.csv


{
  'model': 'csebuetnlp/mT5_multilingual_XLSum',
  'rouge1': 64.54,
  'rouge2': 58.5,
  'rougeL': 63.19,
  'zero_shot_rouge_first100': {'rouge1': 36.84, 'rouge2': 25.17, 'rougeL': 32.68},
  'train_size': 5000,
  'test_size': 500,
  'epochs': 2,
  'lr': 0.0001,
  'max_input': 512,
  'num_beams': 4
}

### การบันทึกบทสรุปทั้งหมด (`predictions_mt5.csv`)
* ส่งออกไฟล์ `predictions_mt5.csv` ประกอบด้วยคอลัมน์ `body`, `summary`, `pred`, `rougeL`
* ส่งต่อไฟล์นี้ให้:
  * **คนที่ 6:** เพื่อตรวจสอบความถูกต้องของชื่อเฉพาะ (NER: บุคคล, สถานที่, องค์กร) ว่าคงอยู่ครบถ้วนในบทสรุปหรือไม่
  * **คนที่ 7:** เพื่อนำไปวิเคราะห์ข้อผิดพลาด (Error Analysis) ของข่าวที่ได้คะแนน ROUGE-L ต่ำ

In [ ]:
import os
import pandas as pd
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

if 'scorer' not in globals():
    class ThaiTokenizer:
        def tokenize(self, text):
            return [w for w in word_tokenize(text, engine="newmm") if w.strip()]
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

if 'test_df' not in globals():
    pred_path = "predictions_mt5.csv" if os.path.exists("predictions_mt5.csv") else ("../predictions_mt5.csv" if os.path.exists("../predictions_mt5.csv") else None)
    if pred_path:
        test_df = pd.read_csv(pred_path)
    else:
        test_df = load(TEST_CSV, "test", TEST_SIZE)

out = test_df.assign(pred=preds)
if "rougeL" not in out.columns or out["rougeL"].isnull().any():
    out["rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
out.to_csv("predictions_mt5.csv", index=False)
pd.set_option("display.max_colwidth", 200)
print("บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:")
out.sort_values("rougeL")[["summary", "pred", "rougeL"]].head()

บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:


,summary,pred,rougeL
321,เห็น แอฟ-ทักษอร ทำหน้าที่ คุณแม่ แข็งขัน พา น้องปีใหม่ ไปไหนมาไหนด้วยตลอด ล่าสุดเจ้าตัวเผยถึงกระแสข่าวหย่ากับสามี สงกรานต์ เตชะณรงค์ ว่าไม่ขอพูดถึงเรื่องนี้,แอฟ-ทักษอร เตชะณรงค์ เผยถึงเรื่องราวความรักกับอดีตสามี สงกรานต์ เตชะณรงค์ พร้อมเปิดใจถึงเรื่องหัวใจว่าไม่ได้เจอกันมานานแล้ว,0.000000
133,ศ.ดร.สุรเกียรติ์ เสถียรไทย ผู้บริหารวุฒิศักดิ์ คลินิก กรุ๊ป จัดงานเปิดตัวแคมเปญใหม่ มั่นใจใช้ชีวิต พลิกโฉมองค์กรสู่การเติบโตอย่างยั่งยืน,"วุฒิศักดิ์ คลินิก กรุ๊ป เปิดตัวแคมเปญ ""มั่นใจใช้ชีวิต"" เพื่อเสริมสร้างความมั่นใจให้กับคนไทยทุกคน",0.000000
371,ชเวซุนซิล คนสนิทของ ปธน.ปาร์กกึนเฮ ปฏิเสธทุกข้อกล่าวหาในคดีอื้อฉาวทุจริตและการใช้อิทธิพลแทรกแซงการเมือง,นางชเว ซุน-ซิล คนสนิทของประธานาธิบดีปาร์ก กึน-เฮ ของเกาหลีใต้ ปฏิเสธทุกข้อกล่าวหาในศาลเกาหลีใต้,0.090909


## 8. ทดสอบสรุปข่าวจริงเหตุการณ์ใหม่ (Inference on Unseen News)
* ป้อนเนื้อหาข่าวตัวอย่างที่เป็นเหตุการณ์ปัจจุบัน เพื่อดูผลการสรุปจริงของโมเดล

In [ ]:
# กำหนดโมเดลที่ใช้งานสรุป
active_model = eval_model if 'eval_model' in globals() else (trainer.model if 'trainer' in globals() and trainer is not None else AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device))

news = """กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ระหว่างวันที่ 10-12 นี้
จะมีพายุฝนฟ้าคะนอง ลมกระโชกแรง และลูกเห็บตกบางพื้นที่ ขอให้ประชาชนหลีกเลี่ยงการอยู่ในที่โล่งแจ้ง
และใต้ต้นไม้ใหญ่ ส่วนเกษตรกรควรเตรียมป้องกันความเสียหายของผลผลิต"""
summarize([news], active_model)[0]

'กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ช่วงวันที่ 10-12 นี้'

## 9. การบันทึกและบรรจุโมเดล (Save & Package Artifacts)
* บันทึก Weights, Config, และ Tokenizer ลงโฟลเดอร์ `mt5-thaisum/`
* บันทึกคะแนนผลการทดสอบลง `result.json`
* บีบอัดเป็นไฟล์ `mt5-thaisum.zip` เพื่อส่งต่อหรือใช้งานต่อไป

In [ ]:
import json, shutil, os

# บันทึกโมเดล (รองรับทั้งกรณีเทรนผ่าน trainer หรือโหลดโมเดล)
if 'trainer' in globals() and trainer is not None and hasattr(trainer, 'save_model'):
    trainer.save_model(OUTPUT_DIR)
elif 'eval_model' in globals():
    eval_model.save_pretrained(OUTPUT_DIR)
elif 'active_model' in globals():
    active_model.save_pretrained(OUTPUT_DIR)

if 'tokenizer' in globals():
    tokenizer.save_pretrained(OUTPUT_DIR)

if 'result' in globals():
    json.dump(result, open(f"{OUTPUT_DIR}/result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("บันทึกแล้ว:", OUTPUT_DIR + ".zip")

# คัดลอกเข้า Drive (เอาคอมเมนต์ออกถ้าต้องการ)
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(OUTPUT_DIR + ".zip", "/content/drive/MyDrive/")

บันทึกแล้ว: mt5-thaisum.zip


## 10. สรุปภาพรวมโครงงาน (Executive Summary & Reflection)

---

### 📌 1. ชื่อเรื่อง (Project Title)
**"ระบบสรุปข่าวภาษาไทยอัตโนมัติด้วยโมเดลภาษา mT5 และการวิเคราะห์ชื่อเฉพาะ"**  
*(Abstractive Thai News Summarization using Fine-tuned mT5 with Named Entity Recognition)*  
**ผู้จัดทำ:** ส่วนที่ 4 · โมเดลหลัก mT5 (คู่ C - คนที่ 4)

---

### 📝 2. คำอธิบายโครงงานและกระบวนการทำงาน (Description & Methodology)
โครงงานนี้พัฒนาขึ้นเพื่อแก้ไขข้อจำกัดของการสรุปข่าวแบบเดิม (Extractive เช่น Lead-3 หรือ TextRank) ที่ทำได้เพียงคัดลอกประโยคเดิมมาวาง ทำให้บทสรุปยาวเกินไปและขาดความต่อเนื่อง โดยเปลี่ยนมาใช้การสรุปเชิงสร้างสรรค์ (Abstractive Summarization) ซึ่งมีรายละเอียดการดำเนินงานดังนี้:
1. **ข้อมูลที่ใช้:** คลังข้อมูลข่าวภาษาไทย **ThaiSum Dataset** จำนวนกว่า 350,000 ข่าว สุ่มใช้สำหรับ Train 5,000 ข่าว, Validation 300 ข่าว และ Test 500 ข่าว โดยโหลดแบบ Streaming เพื่อประสิทธิภาพสูงสุด
2. **โมเดลสถาปัตยกรรม:** ใช้โมเดล **mT5 (Multilingual Text-to-Text Transfer Transformer)** ผ่านโมเดลตั้งต้น `csebuetnlp/mT5_multilingual_XLSum` ขนาดประมาณ 580 ล้านพารามิเตอร์ ซึ่งมีพรีเทรนภาษาไทยและภาษาอื่นรวม 45 ภาษา
3. **การฝึกฝน (Fine-tuning):** ปรับจูนด้วย **Adafactor Optimizer** อัตราการเรียนรู้ `1e-4`, จัดการ Gradient Accumulation เพื่อให้ได้ Batch Size เสมือนเท่ากับ 16, และใช้ **Gradient Checkpointing** เพื่อประหยัด VRAM บน GPU ของ Google Colab
4. **การประเมินผล:** ใช้เกณฑ์มาตรฐาน **ROUGE-1, ROUGE-2, ROUGE-L** โดยตัดคำภาษาไทยด้วย **PyThaiNLP (เอนจิน newmm)** ก่อนคำนวณคะแนน

**ผลการทดสอบเชิงปริมาณ (Test Set 500 ข่าว):**
* **ก่อนเทรน (Zero-shot บน 100 ข่าวแรก):** ROUGE-1 = 36.84% | ROUGE-2 = 25.17% | **ROUGE-L = 32.68%**
* **หลัง Fine-tune (โมเดล mT5 สำเร็จ):** ROUGE-1 = 64.54% | ROUGE-2 = 58.50% | **ROUGE-L = 63.19%**
* *ผลลัพธ์แสดงให้เห็นว่าคะแนน ROUGE-L เพิ่มขึ้นถึง +30.51% (เพิ่มขึ้นเกือบ 1 เท่าตัว)*

---

### 💬 3. ความคิดเห็นและบทวิเคราะห์ผลลัพธ์ (Comments, Error Analysis & Insights)

#### 3.1 ความคิดเห็นต่อประสิทธิภาพของโมเดล (Model Performance Comments):
* โมเดล mT5 สามารถจับใจความสำคัญของข่าวภาษาไทยได้อย่างแม่นยำ สามารถสังเคราะห์ข้อความใหม่ที่สั้นกระชับ (2–3 ประโยค) และคงเนื้อหาสำคัญได้ตรงตามหลักการทำข่าว
* ในหมวดข่าวการเมืองและข่าวเศรษฐกิจ โมเดลทำคะแนนได้ดีเยี่ยม (ROUGE-L อยู่ระหว่าง 70% – 85%) เนื่องจากข่าวกลุ่มนี้มีโครงสร้างการเขียนแบบ **พีระมิดหัวกลับ (Inverted Pyramid)** ซึ่งสาระสำคัญและตัวเลขสถิติจะถูกระบุอย่างชัดเจนตั้งแต่ย่อหน้าแรกๆ

#### 3.2 ความคิดเห็นต่อข้อจำกัดและข้อผิดพลาด (Error Analysis & Limitations):
* **ข่าวที่ได้คะแนน ROUGE-L ต่ำ (เช่น ข่าวบันเทิง/สัมภาษณ์ดารา):**
  * ข่าวกลุ่มนี้มักใช้ภาษาพูด แสลง หรือคำอุปมาอุปไมย (เช่น *"ไม่ขอพูดถึงเรื่องนี้"*, *"ศึกษาดูใจ"*) ทำให้นักข่าวที่เขียนสรุปอ้างอิง และโมเดล mT5 สรุปข้อเท็จจริงออกมาคนละมุมมอง ส่งผลให้คำศัพท์ไม่ตรงกัน แม้ใจความข่าวจะสื่อไปในทางเดียวกัน
* **การตกหล่นของชื่อเฉพาะ (Entity Omission):**
  * ในข่าวที่มีตัวละครหรือหน่วยงานมากกว่า 4–5 รายการ โมเดลมีแนวโน้มจะคัดเลือกเฉพาะ 1–2 ชื่อที่สำคัญที่สุด ทำให้ชื่อเฉพาะรองบางชื่อไม่ปรากฏในบทสรุป ซึ่งส่งผลกระทบต่อคะแนน ROUGE เมื่อเทียบกับบทสรุปยาวของนักข่าว

#### 3.3 ความคิดเห็นและข้อเสนอแนะในการพัฒนาต่อยอด (Future Recommendations):
1. **การเพิ่มขนาดชุดข้อมูล:** หากมีทรัพยากร GPU สูงขึ้น ควรขยายข้อมูลเทรนเป็น 20,000–50,000 ข่าว เพื่อให้โมเดลคุ้นเคยกับคำศัพท์เฉพาะทางและการสรุปในหลากหลายบริบทมากขึ้น
2. **การผสานเทคนิค NER-Guided Decoding:** ควรเพิ่มเงื่อนไขในกระบวนการ Beam Search เพื่อบังคับให้โมเดลคงชื่อเฉพาะสำคัญ (เช่น บุคคลสำคัญ หรือ องค์กรหลัก) ที่ปรากฏในข่าวไว้ในบทสรุปเสมอ
3. **การนำเทคนิค PEFT / LoRA มาใช้:** ช่วยให้สามารถ fine-tune โมเดลขนาดใหญ่กว่านี้ เช่น mT5-base หรือ mT5-large โดยใช้หน่วยความจำ VRAM น้อยลงอย่างมาก